In [1]:
import math
import random
import numpy as np
import torch
from torch import nn


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def true_function(x):
    return torch.sin(2.0 * x) + 0.3 * torch.cos(5.0 * x)


def make_dataset(n_train=256, n_test=256, noise_std=0.1):
    x_train = torch.linspace(-3.0, 3.0, n_train).unsqueeze(1)
    y_clean_train = true_function(x_train)
    y_train = y_clean_train + noise_std * torch.randn_like(y_clean_train)

    x_test = torch.linspace(-3.2, 3.2, n_test).unsqueeze(1)
    y_clean_test = true_function(x_test)
    y_test = y_clean_test + noise_std * torch.randn_like(y_clean_test)

    return (
        x_train.to(device),
        y_train.to(device),
        x_test.to(device),
        y_test.to(device),
        y_clean_test.to(device),
    )


class BayesianSelfRegMLP(nn.Module):
    def __init__(self, in_dim=1, hidden_dim=32, out_dim=1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

    def weight_energy(self):
        ew = torch.tensor(0.0, device=next(self.parameters()).device)
        for p in self.parameters():
            ew = ew + 0.5 * torch.sum(p ** 2)
        return ew

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


@torch.no_grad()
def flatten_tensors(tensors):
    return torch.cat([t.reshape(-1) for t in tensors])


def gauss_newton_diag(model, x):
    """
    Приближённая диагональ Gauss-Newton матрицы для задачи регрессии.

    Для ошибки:
        E_D = 1/2 * sum_i (f(x_i) - y_i)^2

    Gauss-Newton:
        H ~= J^T J

    Диагональ:
        diag(H)_j = sum_i (df_i / dw_j)^2
    """

    params = [p for p in model.parameters() if p.requires_grad]
    h_diag = [torch.zeros_like(p) for p in params]

    model.eval()

    for i in range(x.shape[0]):
        model.zero_grad(set_to_none=True)

        out = model(x[i:i + 1])
        scalar_out = out.squeeze()

        grads = torch.autograd.grad(
            scalar_out,
            params,
            retain_graph=False,
            create_graph=False,
            allow_unused=False,
        )

        for h, g in zip(h_diag, grads):
            h += g.detach() ** 2

    return flatten_tensors(h_diag)


def estimate_gamma(model, x_train, alpha, beta):
    """
    gamma = sum_j beta * lambda_j / (alpha + beta * lambda_j)

    Здесь lambda_j приближённо берём как диагональные элементы J^T J.
    """

    h_diag = gauss_newton_diag(model, x_train)

    gamma_terms = beta * h_diag / (alpha + beta * h_diag + 1e-12)
    gamma = torch.sum(gamma_terms)

    return gamma


def train():
    noise_std = 0.1

    x_train, y_train, x_test, y_test, y_clean_test = make_dataset(
        n_train=256,
        n_test=256,
        noise_std=noise_std,
    )

    model = BayesianSelfRegMLP(
        in_dim=1,
        hidden_dim=32,
        out_dim=1,
    ).to(device)

    n_train = x_train.shape[0]
    n_params = model.num_parameters()

    print(f"Device: {device}")
    print(f"Train samples: {n_train}")
    print(f"Model parameters: {n_params}")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Важно: alpha стартует маленьким.
    # Иначе сеть может сразу зажать веса.
    alpha = torch.tensor(1e-3, device=device)
    beta = torch.tensor(1.0, device=device)

    epochs = 7000

    # Не обновляем alpha/beta с первой эпохи.
    # Сначала даём сети немного научиться.
    warmup_epochs = 500

    hyper_update_every = 100
    hyper_smooth = 0.3

    alpha_min = 1e-8
    alpha_max = 1e4

    beta_min = 1e-8
    beta_max = 1e4

    alpha_history = []
    beta_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        y_pred = model(x_train)

        data_energy = 0.5 * torch.sum((y_pred - y_train) ** 2)
        weight_energy = model.weight_energy()

        loss = beta.detach() * data_energy + alpha.detach() * weight_energy

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=100.0)
        optimizer.step()

        # -----------------------------
        # Обновление alpha/beta
        # -----------------------------
        if epoch > warmup_epochs and epoch % hyper_update_every == 0:
            with torch.no_grad():
                y_pred_new = model(x_train)
                data_energy_new = 0.5 * torch.sum((y_pred_new - y_train) ** 2)
                weight_energy_new = model.weight_energy()

            gamma = estimate_gamma(
                model=model,
                x_train=x_train,
                alpha=alpha.detach(),
                beta=beta.detach(),
            )

            # Из-за диагонального приближения gamma иногда может быть чуть больше N.
            # Для формулы beta нужно gamma < N.
            gamma = torch.clamp(gamma, min=1e-6, max=0.95 * n_train)

            with torch.no_grad():
                alpha_hat = gamma / (2.0 * weight_energy_new + 1e-12)
                beta_hat = (n_train - gamma) / (2.0 * data_energy_new + 1e-12)

                alpha_hat = torch.clamp(alpha_hat, alpha_min, alpha_max)
                beta_hat = torch.clamp(beta_hat, beta_min, beta_max)

                alpha = (1.0 - hyper_smooth) * alpha + hyper_smooth * alpha_hat
                beta = (1.0 - hyper_smooth) * beta + hyper_smooth * beta_hat

        with torch.no_grad():
            model.eval()

            train_pred = model(x_train)
            test_pred = model(x_test)

            train_rmse = torch.sqrt(torch.mean((train_pred - y_train) ** 2))
            test_rmse = torch.sqrt(torch.mean((test_pred - y_test) ** 2))
            clean_test_rmse = torch.sqrt(torch.mean((test_pred - y_clean_test) ** 2))

        alpha_history.append(alpha.item())
        beta_history.append(beta.item())

        if epoch % 500 == 0 or epoch == 1:
            print(
                f"epoch={epoch:5d} | "
                f"loss={loss.item():10.4f} | "
                f"alpha={alpha.item():10.4f} | "
                f"beta={beta.item():10.4f} | "
                f"train_rmse={train_rmse.item():.5f} | "
                f"test_rmse={test_rmse.item():.5f} | "
                f"clean_test_rmse={clean_test_rmse.item():.5f}"
            )

    alpha_history = np.array(alpha_history)
    beta_history = np.array(beta_history)

    def convergence_report(values, name, window=1000):
        tail = values[-window:]
        mean = tail.mean()
        std = tail.std()
        rel_std = std / (abs(mean) + 1e-12)
        drift = abs(tail[-1] - tail[0]) / (abs(mean) + 1e-12)

        print()
        print(f"{name} convergence report:")
        print(f"  last value      : {tail[-1]:.6f}")
        print(f"  mean last window: {mean:.6f}")
        print(f"  std last window : {std:.6f}")
        print(f"  relative std    : {rel_std:.6f}")
        print(f"  relative drift  : {drift:.6f}")

    convergence_report(alpha_history, "alpha")
    convergence_report(beta_history, "beta")

    print()
    print("Final results:")
    print(f"  alpha final       = {alpha_history[-1]:.6f}")
    print(f"  beta final        = {beta_history[-1]:.6f}")
    print(f"  estimated noise σ = {1.0 / math.sqrt(beta_history[-1]):.6f}")
    print(f"  true noise σ      = {noise_std:.6f}")

    return model, alpha_history, beta_history


if __name__ == "__main__":
    train()


Device: cpu
Train samples: 256
Model parameters: 1153
epoch=    1 | loss=   75.2836 | alpha=    0.0010 | beta=    1.0000 | train_rmse=0.75798 | test_rmse=0.73690 | clean_test_rmse=0.73012
epoch=  500 | loss=    5.9716 | alpha=    0.0010 | beta=    1.0000 | train_rmse=0.21505 | test_rmse=0.28885 | clean_test_rmse=0.25976
epoch= 1000 | loss=   55.2339 | alpha=   10.9220 | beta=    1.0091 | train_rmse=0.42844 | test_rmse=0.48928 | clean_test_rmse=0.47437
epoch= 1500 | loss=  107.9157 | alpha=   19.7306 | beta=    1.9591 | train_rmse=0.44247 | test_rmse=0.51054 | clean_test_rmse=0.49618
epoch= 2000 | loss=  123.3741 | alpha=   22.1340 | beta=    2.2042 | train_rmse=0.44379 | test_rmse=0.51407 | clean_test_rmse=0.49985
epoch= 2500 | loss=  126.0280 | alpha=   19.9313 | beta=    2.6981 | train_rmse=0.40048 | test_rmse=0.47288 | clean_test_rmse=0.45705
epoch= 3000 | loss=  126.6130 | alpha=   15.9697 | beta=    3.9726 | train_rmse=0.32093 | test_rmse=0.39774 | clean_test_rmse=0.37752
epoch= 3